# 下载 结直肠癌 Visium HD / Xenium / 单细胞参考（GSE280318）

默认下载超系列全部公开 supplementary 文件，并额外解析 GSE280314 子系列：CRC Xenium 的表达与 cells 坐标包属于 Series 级附件，仅下载超系列会漏掉这些关键包。另取作者公开元数据。大图像较多，先检查清单。

[GEO 官方记录](https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc=GSE280318)

## 1. 环境

Python ≥3.11。下面安装到当前 notebook 内核；已有依赖可跳过此单元格。
下载与转换共用同目录的 `download_utils.py`、`h5ad_utils.py`，无需先运行其他 notebook。
官方文件保存在 `official/`，清单在 `metadata/`，H5AD 在 `h5ad/`。
支持 HTTP 断点续传；有官方校验值时核验校验值，否则验证传输长度。重复运行会检查并跳过已完成文件。

In [1]:
%pip install -q "anndata>=0.11,<0.13" pandas scipy h5py pyarrow


Note: you may need to restart the kernel to use updated packages.


## 2. 路径与参数

In [1]:
from pathlib import Path
import sys
import pandas as pd

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (HERE, *HERE.parents) if (p / "data" / "download").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("请从 DeepSpatial 项目目录或 data/download 运行。")
sys.path.insert(0, str(PROJECT_ROOT / "data" / "download"))
from download_utils import download_records, save_json

DATA_DIR = PROJECT_ROOT / "data" / "CRC-VisiumHD-GSE280318"
DATA_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_H5AD = False  # 可选：改为 True，在官方文件之外增加 H5AD；不修改原文件。
SAMPLE_IDS = None  # None=全部；也可填写 GSM 样本列表
EXTRACT_ARCHIVES = True
Z_UM = {}  # 跨平台相邻切片没有可靠 Z 时保持空字典
print("数据目录：", DATA_DIR)


数据目录： /Users/huangmao/Downloads/Code/DeepSpatial/data/CRC-VisiumHD-GSE280318


## 3. 从官方 SOFT 元数据解析真实下载地址

In [2]:
from download_utils import geo_records

records = geo_records("GSE280318", DATA_DIR, sample_ids=SAMPLE_IDS)
records += geo_records("GSE280314", DATA_DIR,
                       sample_ids=None if SAMPLE_IDS is None else [s for s in SAMPLE_IDS if s in {"GSM8594564", "GSM8594565", "GSM8594566"}],
                       series_samples={r"P1_?CRC": "GSM8594564", r"P2_?CRC": "GSM8594565", r"P5_?CRC": "GSM8594566"})
# Supplementary files of the SuperSeries and SubSeries overlap: download each URL once.
records = list({r["url"]: r for r in records}.values())
AUTHOR_METADATA = ["SingleCell_MetaData.csv.gz", "DeconvolutionResults_P1CRC.csv.gz", "DeconvolutionResults_P2CRC.csv.gz", "DeconvolutionResults_P5CRC.csv.gz"]
for name in AUTHOR_METADATA:
    records.append({"url": "https://raw.githubusercontent.com/10XGenomics/HumanColonCancer_VisiumHD/main/MetaData/" + name,
                    "path": "official/author_metadata/" + name, "sample": "author_metadata"})
display(pd.DataFrame(records)[["sample", "path", "url"]])
print(f"共 {len(records)} 个文件；长度由下载响应核验，metadata/ 中保留原 GEO 清单。")


已校验，跳过：GSE280318_family.soft.gz
已校验，跳过：GSE280314_family.soft.gz


,sample,path,url
0,GSM8594533,official/GSM8594533/GSM8594533_P1L1_P2CRC_BC1_...,https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM85...
1,GSM8594533,official/GSM8594533/GSM8594533_P1L1_P4CRC_BC2_...,https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM85...
2,GSM8594534,official/GSM8594534/GSM8594534_P1L2_P2CRC_BC1_...,https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM85...
3,GSM8594534,official/GSM8594534/GSM8594534_P1L2_P4CRC_BC2_...,https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM85...
4,GSM8594535,official/GSM8594535/GSM8594535_P1L3_P2CRC_BC1_...,https://ftp.ncbi.nlm.nih.gov/geo/samples/GSM85...
...,...,...,...
140,GSE280314,official/GSE280314/GSE280314_gene_panel.json.gz,https://ftp.ncbi.nlm.nih.gov/geo/series/GSE280...
141,author_metadata,official/author_metadata/SingleCell_MetaData.c...,https://raw.githubusercontent.com/10XGenomics/...
142,author_metadata,official/author_metadata/DeconvolutionResults_...,https://raw.githubusercontent.com/10XGenomics/...
143,author_metadata,official/author_metadata/DeconvolutionResults_...,https://raw.githubusercontent.com/10XGenomics/...


共 145 个文件；长度由下载响应核验，metadata/ 中保留原 GEO 清单。


## 4. 下载并解压

保留官方归档。文件逐一下载支持断点续传；ZIP/TAR 只解压数据，不执行其中的脚本。

In [3]:
download_records(records, DATA_DIR, extract=EXTRACT_ARCHIVES)


已校验，跳过：GSM8594533_P1L1_P2CRC_BC1_count_filtered_feature_bc_matrix.h5
已校验，跳过：GSM8594533_P1L1_P4CRC_BC2_count_filtered_feature_bc_matrix.h5
已校验，跳过：GSM8594534_P1L2_P2CRC_BC1_count_filtered_feature_bc_matrix.h5
已校验，跳过：GSM8594534_P1L2_P4CRC_BC2_count_filtered_feature_bc_matrix.h5
已校验，跳过：GSM8594535_P1L3_P2CRC_BC1_count_filtered_feature_bc_matrix.h5
下载：GSM8594535_P1L3_P4CRC_BC2_count_filtered_feature_bc_matrix.h5，已有 0.0 MiB
  0.01 GiB
下载：GSM8594536_P1L4_P2CRC_BC1_count_filtered_feature_bc_matrix.h5，已有 0.0 MiB
  0.02 GiB
连接中断，保留 .part，重试 1/2：Remote end closed connection without response
下载：GSM8594536_P1L4_P4CRC_BC2_count_filtered_feature_bc_matrix.h5，已有 0.0 MiB
  0.01 GiB
下载：GSM8594537_P2L1_P1CRC_BC1_count_filtered_feature_bc_matrix.h5，已有 0.0 MiB
下载：GSM8594537_P2L1_P2NAT_BC4_count_filtered_feature_bc_matrix.h5，已有 0.0 MiB
  0.01 GiB
下载：GSM8594537_P2L1_P3CRC_BC2_count_filtered_feature_bc_matrix.h5，已有 0.0 MiB
  0.02 GiB
  0.04 GiB
下载：GSM8594537_P2L1_P5CRC_BC3_count_filtered_feature_bc_matrix.h5，已

OSError: 文件未完整下载：217602571 / 8664921120

## 5. 可选：空间表达矩阵 → H5AD

Visium/HD 使用真实 tissue_positions（pixel XY）；Xenium 使用 cells 的 x/y centroid（μm）。条码精确连接，保留作者分类/组成字段，不把 spot 最大占比类型当真实单细胞标签。
每个空间样本一个 H5AD；没有空间坐标的 scRNA 参考保留官方矩阵并记录跳过原因。不虚构 Z，不把相邻平台切片误称同平台连续体积。
设置 `EXPORT_H5AD=True`，并先完成下载/解压。

In [ ]:
EXPORT_H5AD=True
if EXPORT_H5AD:
    from h5ad_utils import export_spatial_downloads
    h5ad_files = export_spatial_downloads(DATA_DIR, z_um=Z_UM)
    print(f"导出 {len(h5ad_files)} 个空间样本 H5AD；详见 metadata/h5ad_skipped.json")
else:
    print("官方文件已保存；EXPORT_H5AD=True 启用可选 H5AD 导出。")
